In [2]:
!pip install -q torch torchaudio psutil gputil transformers jiwer whisper pynvml

In [3]:
!pip install -q datasets openai-whisper

In [4]:
import torch
import torchaudio
import numpy as np
import psutil
import time
from transformers import (
    pipeline,
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    AutoModelForCTC,
    Wav2Vec2Processor,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    SpeechT5Processor,
    SpeechT5ForSpeechToText
)
from datasets import load_dataset
from jiwer import wer
import gc
import warnings
warnings.filterwarnings("ignore")

In [21]:
class STTEvaluator:
    def __init__(self, device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device
        self.models = {
            #'whisper-tiny': ('openai/whisper-tiny', self._load_whisper, self._transcribe_whisper),
            #'whisper-base': ('openai/whisper-base', self._load_whisper, self._transcribe_whisper),
            #'whisper-small': ('openai/whisper-small', self._load_whisper, self._transcribe_whisper),
            #'whisper-medium': ('openai/whisper-medium', self._load_whisper, self._transcribe_whisper),
            #'whisper-large-v2': ('openai/whisper-large-v2', self._load_whisper, self._transcribe_whisper),
            #'hubert-en': ('facebook/hubert-large-ls960-ft', self._load_wav2vec2, self._transcribe_wav2vec2)
#            'wav2vec2-base': ('facebook/wav2vec2-base-960h', self._load_wav2vec2, self._transcribe_wav2vec2),
#            'wav2vec2-large': ('facebook/wav2vec2-large-960h', self._load_wav2vec2, self._transcribe_wav2vec2),
#            'wav2vec2-english': ('addy88/wav2vec2-english-stt', self._load_wav2vec2, self._transcribe_wav2vec2),
#            'wav2vec2-xlsr-en': ('jonatasgrosman/wav2vec2-large-xlsr-53-english', self._load_wav2vec2, self._transcribe_wav2vec2),
#            'wav2vec2-robust': ('facebook/wav2vec2-large-robust', self._load_wav2vec2, self._transcribe_wav2vec2),
            #'wav2vec2-conformer': ('facebook/wav2vec2-conformer-rel-pos-large', self._load_wav2vec2, self._transcribe_wav2vec2),
            #'wav2vec2-xls-r-300m': ('facebook/wav2vec2-xls-r-300m', self._load_wav2vec2, self._transcribe_wav2vec2),
            #'wav2vec2-xls-r-1b': ('facebook/wav2vec2-xls-r-1b', self._load_wav2vec2, self._transcribe_wav2vec2),
            #'wav2vec2-xls-r-2b': ('facebook/wav2vec2-xls-r-2b', self._load_wav2vec2, self._transcribe_wav2vec2),
           # 'wavlm-base': ('microsoft/wavlm-base', self._load_wav2vec2, self._transcribe_wav2vec2),
           # 'wavlm-base-plus': ('microsoft/wavlm-base-plus', self._load_wav2vec2, self._transcribe_wav2vec2),
           # 'data2vec-audio-base': ('facebook/data2vec-audio-base', self._load_wav2vec2, self._transcribe_wav2vec2),
        }
        self.results = {}

    def _get_memory_usage(self):
        if self.device == 'cuda':
            gpu_memory = torch.cuda.memory_allocated() / (1024 * 1024)  # Convert to MB
        else:
            gpu_memory = 0
        cpu_memory = psutil.Process().memory_info().rss / (1024 * 1024)  # Convert to MB
        return cpu_memory, gpu_memory

    def _resample_audio(self, waveform, original_sr, target_sr):
        if original_sr != target_sr:
            resampler = torchaudio.transforms.Resample(original_sr, target_sr)
            return resampler(waveform)
        return waveform

    def _load_whisper(self, model_id):
        try:
            processor = WhisperProcessor.from_pretrained(model_id)
            model = WhisperForConditionalGeneration.from_pretrained(model_id).to(self.device)
            return model, processor
        except Exception as e:
            print(f"Error loading Whisper model: {str(e)}")
            raise

    def _load_wav2vec2(self, model_id):
        try:
            processor = Wav2Vec2Processor.from_pretrained(model_id)
            model = AutoModelForCTC.from_pretrained(model_id).to(self.device)
            return model, processor
        except Exception as e:
            print(f"Error loading Wav2Vec2 model: {str(e)}")
            raise

    def _transcribe_whisper(self, model, processor, audio_path, waveform, sample_rate):
        waveform = self._resample_audio(waveform, sample_rate, 16000)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        input_features = processor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors="pt").input_features.to(self.device)

        with torch.no_grad():
            predicted_ids = model.generate(input_features)
            transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

        return transcription

    def _transcribe_wav2vec2(self, model, processor, audio_path, waveform, sample_rate):
        waveform = self._resample_audio(waveform, sample_rate, 16000)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        input_values = processor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors="pt").input_values.to(self.device)

        with torch.no_grad():
            logits = model(input_values).logits
            predicted_ids = torch.argmax(logits, dim=-1)
            transcription = processor.batch_decode(predicted_ids)[0]

        return transcription

    def evaluate_model(self, model_name, audio_path, reference_text):
        print(f"\nEvaluating {model_name}...")
        results = {}

        try:
            # Load audio and get duration
            waveform, sample_rate = torchaudio.load(audio_path)
            audio_duration = waveform.shape[1] / sample_rate
            print(f"Audio duration: {audio_duration:.2f} seconds")

            # Memory cleanup
            torch.cuda.empty_cache()
            gc.collect()
            initial_cpu_mem, initial_gpu_mem = self._get_memory_usage()

            # Load model
            start_time = time.time()
            model_id, load_func, transcribe_func = self.models[model_name]
            print(f"Loading model: {model_id}")
            model, processor = load_func(model_id)
            loading_time = time.time() - start_time

            # Get memory after loading
            load_cpu_mem, load_gpu_mem = self._get_memory_usage()

            # Inference
            start_time = time.time()
            transcription = transcribe_func(model, processor, audio_path, waveform, sample_rate)
            inference_time = time.time() - start_time

            # Calculate metrics
            rtf = inference_time / audio_duration
            inference_speed = 1 / rtf if rtf > 0 else 0

            print(f"Transcription: {transcription}")

            results = {
                'wer': wer(reference_text, transcription),
                'rtf': rtf,
                'latency': inference_time,
                'cpu_memory_mb': load_cpu_mem - initial_cpu_mem,
                'gpu_memory_mb': load_gpu_mem - initial_gpu_mem,
                'inference_speed': inference_speed,
                'loading_time': loading_time,
                'transcription': transcription,
                'status': 'success'
            }

        except Exception as e:
            print(f"Error evaluating {model_name}: {str(e)}")
            results = {
                'status': 'failed',
                'error': str(e)
            }

        # Cleanup
        if self.device == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()

        self.results[model_name] = results
        return results

    def evaluate_all(self, audio_path, reference_text):
        for model_name in self.models.keys():
            self.evaluate_model(model_name, audio_path, reference_text)
        return self.results

    def print_results(self):
        print("\nEvaluation Results:")
        print("-" * 150)
        headers = ['Model', 'WER', 'RTF', 'Latency(s)', 'CPU Mem(MB)', 'GPU Mem(MB)', 'Speed', 'Status']
        print(f"{headers[0]:<25} {headers[1]:<10} {headers[2]:<10} {headers[3]:<12} {headers[4]:<12} "
              f"{headers[5]:<12} {headers[6]:<10} {headers[7]:<10}")
        print("-" * 150)

        for model_name, results in self.results.items():
            if results['status'] == 'success':
                print(f"{model_name:<25} {results['wer']:<10.3f} {results['rtf']:<10.3f} "
                      f"{results['latency']:<12.3f} {results['cpu_memory_mb']:<12.1f} "
                      f"{results['gpu_memory_mb']:<12.1f} {results['inference_speed']:<10.2f} "
                      f"{results['status']:<10}")
            else:
                print(f"{model_name:<25} {'N/A':<10} {'N/A':<10} {'N/A':<12} {'N/A':<12} "
                      f"{'N/A':<12} {'N/A':<10} {results['status']:<10}")
                print(f"Error: {results['error']}")

In [22]:
# Initialize evaluator
evaluator = STTEvaluator()

In [23]:
# Test with your audio file
audio_path = "/content/harvard.wav"
reference_text = """The stale smell of old beer lingers. It takes heat to bring out the odor. A cold dip restores health and zest. A salt pickle tastes fine with ham. Tacos al pastor are my favorite. A zestful food is the hot cross bun."""

all_results = evaluator.evaluate_all(audio_path, reference_text)

# Print results
evaluator.print_results()


Evaluating wavlm-base...
Audio duration: 18.36 seconds
Loading model: microsoft/wavlm-base


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.24k [00:00<?, ?B/s]

Error loading Wav2Vec2 model: Can't load tokenizer for 'microsoft/wavlm-base'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'microsoft/wavlm-base' is the correct path to a directory containing all relevant files for a Wav2Vec2CTCTokenizer tokenizer.
Error evaluating wavlm-base: Can't load tokenizer for 'microsoft/wavlm-base'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'microsoft/wavlm-base' is the correct path to a directory containing all relevant files for a Wav2Vec2CTCTokenizer tokenizer.

Evaluating wavlm-base-plus...
Audio duration: 18.36 seconds
Loading model: microsoft/wavlm-base-plus


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.23k [00:00<?, ?B/s]

Error loading Wav2Vec2 model: Can't load tokenizer for 'microsoft/wavlm-base-plus'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'microsoft/wavlm-base-plus' is the correct path to a directory containing all relevant files for a Wav2Vec2CTCTokenizer tokenizer.
Error evaluating wavlm-base-plus: Can't load tokenizer for 'microsoft/wavlm-base-plus'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'microsoft/wavlm-base-plus' is the correct path to a directory containing all relevant files for a Wav2Vec2CTCTokenizer tokenizer.

Evaluating data2vec-audio-base...
Audio duration: 18.36 seconds
Loading model: facebook/data2vec-audio-base


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

Error loading Wav2Vec2 model: Can't load tokenizer for 'facebook/data2vec-audio-base'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'facebook/data2vec-audio-base' is the correct path to a directory containing all relevant files for a Wav2Vec2CTCTokenizer tokenizer.
Error evaluating data2vec-audio-base: Can't load tokenizer for 'facebook/data2vec-audio-base'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'facebook/data2vec-audio-base' is the correct path to a directory containing all relevant files for a Wav2Vec2CTCTokenizer tokenizer.

Evaluation Results:
------------------------------------------------------------------------------------------------------------------------------------------------------
Model                     WER        RTF        Latency(s)   CPU Mem(MB)  